# MobileNetV2 — 3 thí nghiệm, Resize + Padding 224×224

**Mục đích:** Đánh giá đầy đủ **từng** phiên bản MobileNetV2 (Frozen, Fine-tuning, Fine-tuning + Train Augmentation), **không xếp hạng và không chọn một mô hình duy nhất**.

- Dùng đúng `manifest_used.csv` của nhóm: **2.457 ảnh**, `train=1.716`, `valid=363`, `test=378`, có kiểm tra tách `source_id`.
- Tiền xử lý hình học thống nhất: **giữ tỷ lệ ảnh rồi padding vào khung 224×224 (letterbox)**; **không crop ảnh**. ImageNet normalization chỉ phục vụ MobileNetV2. Dùng RGB cả ba thí nghiệm.
- Fine-tuning: mở các tầng `features[15:19]` và classifier, đúng **1.531.204** tham số cập nhật (TorchVision width_mult=1.0); Frozen chỉ cập nhật classifier **5.124** tham số.
- Augmentation của thí nghiệm 3 **chỉ áp dụng khi huấn luyện**, không dùng thay đổi màu mạnh vì màu là tín hiệu quan trọng của độ chín.
- **Mỗi thí nghiệm có checkpoint tốt nhất theo Validation F1-Macro và checkpoint epoch cuối**, nhưng **không có bất kỳ phép chọn mô hình tốt nhất giữa 3 thí nghiệm**. Test đánh giá riêng từng thí nghiệm bằng best-validation checkpoint *của chính nó*, tránh chọn theo Test.
- Xuất đủ train history, loss/accuracy/F1 curves, macro/weighted/per-class Precision–Recall–F1, Test/Valid confusion matrix (số lượng và chuẩn hóa), dự đoán + xác suất 4 lớp từng ảnh, feature maps, Grad-CAM, ablation RGB→grayscale. **Không tạo ZIP output**.

### Chuẩn bị Kaggle Add Input

1. **Add Input** bộ ảnh gốc có tên tệp tương ứng với các đường dẫn trong manifest, ví dụ `Green/Green_106_jpg.rf.37bdd4c24e648dfd445d0c9b788b00db.jpg`. Notebook tự tìm ảnh trong các thư mục đầu vào hoặc trong ZIP được Add Input.
2. **Add Input** pretrained ImageNet cho TorchVision MobileNetV2 khi Kaggle tắt Internet: tệp `mobilenet_v2-b0353104.pth`. Nếu có Internet, notebook thử tải từ TorchVision khi không tìm thấy tệp này.
3. **Manifest đã được nhúng trực tiếp vào notebook**, nên không bắt buộc Add Input manifest. Notebook sẽ xuất lại một bản `manifest_used.csv` để kiểm toán.
4. Khuyến nghị chạy GPU và chọn **Save Version → Save & Run All (Commit)**. Toàn bộ file nằm trong `/kaggle/working/banana_mobilenet_letterbox_results/`; không nén lại.

> **Lưu ý về độ chính xác:** Đây là **thí nghiệm mới** do thay đổi tiền xử lý từ resize trực tiếp sang letterbox. Không mặc định kỳ vọng tái tạo các mức Accuracy cũ 93,12% / 94,44% / 94,71%; cần chạy và đánh giá kết quả mới.

In [ ]:
import os, sys, io, json, re, time, math, random, shutil, hashlib, warnings, zipfile, gzip, base64, copy
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
MAX_EPOCHS = 25
EARLY_STOP_PATIENCE = 7
HEAD_LR_FROZEN = 1e-3
HEAD_LR_FINETUNE = 5e-4
BACKBONE_LR_FINETUNE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
USE_AMP = True
RUN_FEATURE_MAPS = True
RUN_GRADCAM = True
RUN_GRAYSCALE_ABLATION = True
ALLOW_ZIP_INPUT = True
MANUAL_PRETRAINED_WEIGHTS = ''  # Nếu cần: '/kaggle/input/your-weights/mobilenet_v2-b0353104.pth'
MANUAL_IMAGE_ROOTS = []       # Ví dụ: ['/kaggle/input/banana-dataset']
CLASS_NAMES = ['Green', 'Semi-ripe', 'Ripe', 'Overripe']
PAD_COLOR = (128, 128, 128)
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

BASE_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUTPUT_BASE = BASE_DIR / 'banana_mobilenet_letterbox_results'
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT = OUTPUT_BASE / f'run_{RUN_ID}'
OUT.mkdir(parents=True, exist_ok=False)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
print('Torch:', torch.__version__, '| Device:', DEVICE, '| Output:', OUT)
print('Experiments run independently; all 3 results will be saved, with no winner selection.')

## 1. Manifest cố định — kiểm tra chia dữ liệu, chống rò rỉ theo `source_id`

Manifest được nhúng từ tệp `manifest_used.csv` đã dùng trong đồ án. Mọi kết quả mới đều kèm manifest và SHA-256 để nhóm kiểm chứng cùng split; không tạo chia train/valid/test mới.

In [ ]:
# Gzip + Base64 của đúng manifest 2.457 dòng do nhóm cung cấp (nhúng để notebook chạy độc lập).
EMBEDDED_MANIFEST_B64 = (
    '...'
)
EXPECTED_MANIFEST_SHA256 = '2ea3def7e0fdc7718abd7dde904c0946ddd205ef6e0a67f86090337596e5f176'
manifest_bytes = gzip.decompress(base64.b64decode(EMBEDDED_MANIFEST_B64))
assert hashlib.sha256(manifest_bytes).hexdigest() == EXPECTED_MANIFEST_SHA256
manifest_file = OUT / 'manifest_used.csv'
manifest_file.write_bytes(manifest_bytes)
manifest = pd.read_csv(io.BytesIO(manifest_bytes))

assert set(['path','label','source_id','split','label_id']).issubset(manifest.columns)
assert len(manifest) == 2457, f'Manifest không đúng 2457 ảnh: {len(manifest)}'
assert set(manifest.split.unique()) == {'train','valid','test'}
assert manifest.groupby('split').size().to_dict() == {'train':1716,'valid':363,'test':378}
assert manifest['path'].is_unique
assert set(manifest['label']) == set(CLASS_NAMES)
assert all(manifest.label_id.astype(int) == manifest.label.map({c:i for i,c in enumerate(CLASS_NAMES)}))
assert manifest.groupby('source_id').split.nunique().max() == 1, 'Phát hiện trùng source_id giữa các split!'

split_table = pd.crosstab(manifest['split'], manifest['label']).reindex(
    index=['train','valid','test'], columns=CLASS_NAMES, fill_value=0)
split_table['Total'] = split_table.sum(axis=1)
split_table.to_csv(OUT / 'split_distribution.csv', encoding='utf-8-sig')
print('SHA-256 manifest:',EXPECTED_MANIFEST_SHA256)
print(split_table.to_string())

## 2. Tự tìm ảnh từ Kaggle Add Input

Đầu tiên tìm ảnh giải nén trong `/kaggle/input`. Nếu thiếu, tìm trong ZIP đã được Add Input rồi **chỉ giải nén những ảnh nằm trong manifest** sang thư mục làm việc. Không tự thay nhãn, không tự ghép ảnh khác khi thiếu.

In [ ]:
IMAGE_EXTS = {'.jpg','.jpeg','.png','.bmp','.webp'}
NEEDED = set((str(r.label).lower(), Path(str(r.path)).name.lower()) for r in manifest.itertuples())

def search_roots():
    roots = [Path(p) for p in MANUAL_IMAGE_ROOTS]
    roots += [Path('/kaggle/input'), Path('/kaggle/working')]
    roots = [p for p in roots if p.is_dir()]
    if not roots:
        roots = [Path('/mnt/data')]
    return roots

def scan_image_files(roots):
    mapping = {}
    archives = []
    for root in roots:
        for parent, dirs, files in os.walk(root):
            here = Path(parent)
            # Không quét outputs mới sinh ra để tránh trùng lặp
            dirs[:] = [d for d in dirs if (here / d).resolve() != OUTPUT_BASE.resolve()
                       and d not in {'.git', '.cache', '__pycache__'}]
            for fname in files:
                full = here / fname
                if full.suffix.lower() == '.zip' and ALLOW_ZIP_INPUT:
                    archives.append(full)
                elif full.suffix.lower() in IMAGE_EXTS:
                    k = (here.name.lower(), fname.lower())
                    if k in NEEDED:
                        # Nếu trùng ảnh ở nhiều vị trí, ưu tiên tệp tìm được đầu tiên
                        mapping.setdefault(k, full)
    return mapping, archives

image_index, zip_archives = scan_image_files(search_roots())
missing_keys = NEEDED.difference(image_index)

if missing_keys and ALLOW_ZIP_INPUT:
    temp_root = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else Path('/tmp')
    extract_dir = temp_root / f'banana_mobilenet_input_cache_{RUN_ID}'
    for zpath in zip_archives:
        if not missing_keys:
            break
        try:
            with zipfile.ZipFile(zpath) as zf:
                for info in zf.infolist():
                    if info.is_dir() or info.filename.startswith('__MACOSX/'):
                        continue
                    rel = Path(info.filename.replace('\\','/'))
                    k = (rel.parent.name.lower(), rel.name.lower())
                    if k not in missing_keys:
                        continue
                    # Không dùng đường dẫn ZIP để trích xuất -> tránh path traversal
                    safe_class, safe_name = k
                    dest = extract_dir / safe_class / safe_name
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    with zf.open(info, 'r') as src, open(dest, 'wb') as dst:
                        shutil.copyfileobj(src, dst)
                    image_index[k] = dest
                    missing_keys.remove(k)
        except (zipfile.BadZipFile, OSError) as e:
            print('Bỏ qua ZIP không đọc được:',zpath.name,':',str(e)[:120])

if missing_keys:
    samples = sorted([f'{cl}/{fn}' for cl,fn in missing_keys])[:12]
    raise FileNotFoundError(
        f'Chưa tìm thấy {len(missing_keys)}/{len(NEEDED)} ảnh tương ứng manifest. '
        f'Hãy Add Input đúng bộ ảnh augmented có tên .rf...jpg. Ví dụ thiếu: {samples}. '
        'Nếu ảnh đã giải nén ở nơi khác, đặt MANUAL_IMAGE_ROOTS ở cell cấu hình.')

manifest = manifest.copy()
manifest['resolved_path'] = [str(image_index[(str(r.label).lower(), Path(str(r.path)).name.lower())])
                             for r in manifest.itertuples()]
print(f'Đã ánh xạ đủ {len(manifest)} ảnh | số ZIP thấy: {len(zip_archives)}')
# Đường dẫn resolved_path chỉ dùng nội bộ runtime, manifest gốc không bị thay đổi.

## 3. Tiền xử lý chung: Resize giữ tỷ lệ + Padding (224×224)

- **Giữ nguyên hình học:** resize theo một tỷ lệ chung cho chiều rộng/cao, ảnh còn lại được padding hai phía tương ứng.
- **Không dùng CenterCrop** làm mất một phần quả chuối.
- **Thí nghiệm 3:** RandomHorizontalFlip + RandomAffine nhẹ trên Train, không dùng ColorJitter/RandomGrayscale để tránh làm sai tín hiệu màu độ chín.
- Tất cả các tập ở cả ba phương pháp có **cùng** phép ResizePad và ImageNet normalization. Validation/Test hoàn toàn deterministic.

In [ ]:
class ResizePad:
    def __init__(self, size=IMAGE_SIZE, fill=PAD_COLOR):
        self.size = int(size)
        self.fill = tuple(fill)
    def __call__(self, img):
        if img.mode != 'RGB':
            img = img.convert('RGB')
        w, h = img.size
        if min(w,h) <= 0:
            raise ValueError('Ảnh có kích thước không hợp lệ')
        scale = self.size / max(w,h)
        new_w = min(self.size, max(1, int(round(w*scale))))
        new_h = min(self.size, max(1, int(round(h*scale))))
        resized = img.resize((new_w,new_h), resample=Image.Resampling.BILINEAR)
        canvas = Image.new('RGB', (self.size,self.size), self.fill)
        canvas.paste(resized, ((self.size-new_w)//2,(self.size-new_h)//2))
        return canvas

resize_pad = ResizePad()
train_plain_transform = transforms.Compose([
    resize_pad, transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
train_aug_transform = transforms.Compose([
    resize_pad,
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=10, translate=(0.03,0.03), scale=(0.96,1.0),
                            interpolation=InterpolationMode.BILINEAR, fill=PAD_COLOR),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_transform = train_plain_transform
gray_eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    resize_pad, transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class BananaDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        with Image.open(row.resolved_path) as img:
            tensor = self.transform(img.convert('RGB'))
        return tensor, int(row.label_id), str(row.path)

splits = {s: manifest[manifest.split==s].copy() for s in ['train','valid','test']}
def make_loader(split, transform, shuffle=False):
    ds = BananaDataset(splits[split], transform)
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'),
                      generator=generator, persistent_workers=(NUM_WORKERS>0))

def unnormalize(tensor):
    mean = np.array(MEAN).reshape(3,1,1)
    std = np.array(STD).reshape(3,1,1)
    return np.clip((tensor.detach().cpu().numpy()*std+mean).transpose(1,2,0),0,1)

# Kiểm tra trực quan ảnh trước/sau, lưu để đối chiếu tính chất 'không crop'
fig, axes = plt.subplots(2,4,figsize=(12,6))
for col, row in enumerate(splits['train'].groupby('label',sort=False).head(1).itertuples()):
    if col>=4:break
    with Image.open(row.resolved_path) as im:
        original = im.convert('RGB')
        padded = resize_pad(original)
    axes[0,col].imshow(original)
    axes[0,col].set_title(f'{row.label} • ảnh gốc {original.size}')
    axes[1,col].imshow(padded)
    axes[1,col].set_title(f'Letterbox {IMAGE_SIZE}×{IMAGE_SIZE}')
for ax in axes.flat: ax.axis('off')
fig.tight_layout()
fig.savefig(OUT/'letterbox_examples.png',dpi=150,bbox_inches='tight')
plt.show()

## 4. Trọng số pretrained ImageNet + cấu hình ba mô hình

- **M1:** classifier 4 lớp (5.124 tham số cập nhật).
- **M2/M3:** classifier + `features[15:]` gồm IRB cuối và 1×1 convolution (1.531.204 tham số cập nhật).
- Với những tầng backbone không train, áp dụng chế độ `eval()` để **không cập nhật BatchNorm running stats**.
- Cả ba mô hình **khởi tạo từ cùng pretrained weights**, classifier được seed giống nhau; không tái sử dụng weights đã fine-tune từ mô hình trước.

In [ ]:
def discover_pretrained_weights():
    if MANUAL_PRETRAINED_WEIGHTS:
        p=Path(MANUAL_PRETRAINED_WEIGHTS)
        if not p.exists(): raise FileNotFoundError(f'Không thấy pretrained weights tại: {p}')
        return p
    names=('mobilenet_v2-b0353104.pth',)
    folders=[Path('/kaggle/input'), Path.home()/'.cache/torch/hub/checkpoints',
             Path(torch.hub.get_dir())/'checkpoints', Path('/mnt/data')]
    for folder in folders:
        if not folder.exists(): continue
        for root,_,files in os.walk(folder):
            for filename in files:
                if filename in names:
                    return Path(root)/filename
    return None

def strip_state_dict(obj):
    if isinstance(obj,dict):
        for k in ('state_dict','model_state_dict'):
            if k in obj and isinstance(obj[k],dict): obj=obj[k]
    if not isinstance(obj,dict): raise TypeError('File weights không phải torch state_dict')
    if any(k.startswith('module.') for k in obj.keys()):
        obj = {k.removeprefix('module.'):v for k,v in obj.items()}
    return obj

weight_path = discover_pretrained_weights()
if weight_path is not None:
    base = mobilenet_v2(weights=None)
    try:
        pretrained_obj = torch.load(weight_path, map_location='cpu', weights_only=True)
        base.load_state_dict(strip_state_dict(pretrained_obj), strict=True)
    except Exception as exc:
        raise RuntimeError(
            f'Tệp {weight_path} không phải pretrained MobileNetV2 ImageNet 1000 lớp chuẩn. '
            'Hãy Add Input đúng file mobilenet_v2-b0353104.pth') from exc
    print('Loaded pretrained ImageNet weights from:', weight_path)
else:
    try:
        base = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
        print('Loaded pretrained weights via TorchVision (download/cache)')
    except Exception as exc:
        raise RuntimeError('Không tải được ImageNet pretrained weights. Khi Kaggle tắt Internet, '
                           'hãy Add Input tệp mobilenet_v2-b0353104.pth và chạy lại cell này. '
                           'KHÔNG được tự động fallback sang weights=None.') from exc

BASE_STATE = {k:v.cpu().clone() for k,v in base.state_dict().items()}
del base
EXPERIMENTS = [
    ('mobilenet_1', 'Frozen Backbone', False, False),
    ('mobilenet_2', 'Fine-tuning', True, False),
    ('mobilenet_3', 'Fine-tuning + Augmentation', True, True),
]

def build_model(finetune):
    set_seed(SEED)  # cùng cách khởi tạo classifier
    model = mobilenet_v2(weights=None)
    model.load_state_dict(BASE_STATE, strict=True)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, len(CLASS_NAMES))
    for p in model.features.parameters():
        p.requires_grad_(False)
    if finetune:
        for layer in model.features[15:]:
            for p in layer.parameters(): p.requires_grad_(True)
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    expected = 1531204 if finetune else 5124
    assert total==expected, f'Trainable count {total}, expected {expected}'
    return model

def set_model_train_mode(model, finetune):
    model.train()
    # BN thuộc các block bị freeze phải ở eval (không cập nhật running_mean/var)
    for index,layer in enumerate(model.features):
        if (not finetune) or (index<15):
            layer.eval()

for name,strategy,ft,aug in EXPERIMENTS:
    m = build_model(ft)
    print(name,strategy,'trainable=',sum(p.numel() for p in m.parameters() if p.requires_grad),
          '| online augmentation=',aug)
    del m

## 5. Huấn luyện độc lập từng thí nghiệm

Mỗi epoch xuất: loss, Accuracy, Precision/Recall/F1 Macro, F1 Weighted và F1 theo từng lớp (Train/Validation). Tiêu chí lưu checkpoint **bên trong từng thí nghiệm** là Validation F1-Macro cao nhất, không xếp hạng các thí nghiệm theo Test.

In [ ]:
def calc_summary_metrics(y_true, y_pred, loss=None):
    return {
        'loss': float(loss) if loss is not None else None,
        'accuracy': float(accuracy_score(y_true,y_pred)),
        'precision_macro':float(precision_recall_fscore_support(y_true,y_pred,labels=list(range(4)),average='macro',zero_division=0)[0]),
        'recall_macro':float(precision_recall_fscore_support(y_true,y_pred,labels=list(range(4)),average='macro',zero_division=0)[1]),
        'f1_macro':float(precision_recall_fscore_support(y_true,y_pred,labels=list(range(4)),average='macro',zero_division=0)[2]),
        'precision_weighted':float(precision_recall_fscore_support(y_true,y_pred,labels=list(range(4)),average='weighted',zero_division=0)[0]),
        'recall_weighted':float(precision_recall_fscore_support(y_true,y_pred,labels=list(range(4)),average='weighted',zero_division=0)[1]),
        'f1_weighted':float(precision_recall_fscore_support(y_true,y_pred,labels=list(range(4)),average='weighted',zero_division=0)[2]),
    }

def add_per_class_stats(metrics, y_true, y_pred):
    p,r,f,s = precision_recall_fscore_support(y_true,y_pred,labels=list(range(4)),zero_division=0)
    for i,cl in enumerate(CLASS_NAMES):
        key = re.sub(r'[^a-z0-9]+','_',cl.lower()).strip('_')
        metrics[f'precision_{key}'] = float(p[i])
        metrics[f'recall_{key}'] = float(r[i])
        metrics[f'f1_{key}'] = float(f[i])
        metrics[f'support_{key}'] = int(s[i])
    return metrics

def pass_epoch(model,loader,optimizer=None,scaler=None,finetune=False,collect_predictions=False):
    training = optimizer is not None
    if training: set_model_train_mode(model, finetune)
    else: model.eval()
    loss_fn = nn.CrossEntropyLoss()
    total_loss, n = 0.0,0
    ys, preds, probabilities, filenames = [],[],[],[]
    for x,y,paths in loader:
        x=x.to(DEVICE,non_blocking=True)
        y=y.to(DEVICE,non_blocking=True)
        if training: optimizer.zero_grad(set_to_none=True)
        ctx = torch.autocast(device_type='cuda',dtype=torch.float16) if USE_AMP and DEVICE.type=='cuda' else nullcontext()
        with torch.set_grad_enabled(training):
            with ctx:
                logits=model(x)
                loss=loss_fn(logits,y)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        batch_n=len(y)
        n+=batch_n
        total_loss+=float(loss.item())*batch_n
        probs=torch.softmax(logits.float().detach(),dim=1).cpu().numpy()
        probabilities.extend(probs.tolist())
        ys.extend(y.detach().cpu().tolist())
        preds.extend(np.argmax(probs,axis=1).tolist())
        if collect_predictions: filenames.extend(list(paths))
    stats=calc_summary_metrics(ys,preds,total_loss/max(n,1))
    add_per_class_stats(stats,ys,preds)
    if collect_predictions:
        stats['_y_true']=ys
        stats['_y_pred']=preds
        stats['_probabilities']=probabilities
        stats['_paths']=filenames
    return stats

def optimizer_for(model,finetune):
    if not finetune:
        return torch.optim.AdamW(model.classifier.parameters(),lr=HEAD_LR_FROZEN,weight_decay=WEIGHT_DECAY)
    return torch.optim.AdamW([
        {'params':list(model.features[15:].parameters()), 'lr':BACKBONE_LR_FINETUNE},
        {'params':list(model.classifier.parameters()), 'lr':HEAD_LR_FINETUNE},
    ], weight_decay=WEIGHT_DECAY)

def new_scaler():
    try: return torch.amp.GradScaler('cuda', enabled=(USE_AMP and DEVICE.type=='cuda'))
    except (AttributeError,TypeError):
        return torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE.type=='cuda'))

def save_checkpoint(path,model,epoch,val_metrics,name,strategy,finetune,aug):
    torch.save({'model_state_dict':model.state_dict(),'epoch':int(epoch),
                'val_metrics':{k:float(v) for k,v in val_metrics.items() if isinstance(v,(int,float,np.floating))},
                'experiment':name,'strategy':strategy,'finetune':finetune,'augmentation':aug,
                'classes':CLASS_NAMES,'image_size':IMAGE_SIZE,'preprocessing':'resize_preserve_aspect_pad',
                'manifest_sha256':EXPECTED_MANIFEST_SHA256},path)

def plot_history(history_df,output_path,experiment):
    fig,ax=plt.subplots(1,3,figsize=(16,4))
    for i,(k,ytitle) in enumerate([('loss','Loss'),('accuracy','Accuracy'),('f1_macro','F1-Macro')]):
        ax[i].plot(history_df['epoch'], history_df['train_'+k],marker='.',label='Train')
        ax[i].plot(history_df['epoch'], history_df['valid_'+k],marker='.',label='Validation')
        ax[i].set(title=ytitle,xlabel='Epoch',ylabel=ytitle)
        ax[i].legend()
        ax[i].grid(alpha=.25)
    fig.suptitle(experiment)
    fig.tight_layout()
    fig.savefig(output_path,dpi=160,bbox_inches='tight')
    plt.close(fig)

all_runs=[]
for name,strategy,finetune,augmentation in EXPERIMENTS:
    exp_dir=OUT/name
    exp_dir.mkdir(parents=True,exist_ok=True)
    (exp_dir/'checkpoints').mkdir(exist_ok=True)
    print('\n'+'='*80+'\n'+name,strategy,'\n'+'='*80)
    set_seed(SEED)
    model=build_model(finetune).to(DEVICE)
    train_loader=make_loader('train',train_aug_transform if augmentation else train_plain_transform,shuffle=True)
    valid_loader=make_loader('valid',eval_transform,shuffle=False)
    opt=optimizer_for(model,finetune)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='max',factor=0.5,patience=3)
    scaler=new_scaler()
    hist=[]
    best_f1=-float('inf')
    best_epoch=0
    without_improve=0
    start=time.perf_counter()
    best_path=exp_dir/'checkpoints'/'best_validation_f1.pth'
    last_path=exp_dir/'checkpoints'/'last_epoch.pth'
    for epoch in range(1,MAX_EPOCHS+1):
        train_res=pass_epoch(model,train_loader,optimizer=opt,scaler=scaler,finetune=finetune)
        valid_res=pass_epoch(model,valid_loader)
        scheduler.step(valid_res['f1_macro'])
        row={'epoch':epoch,
             **{'train_'+k:v for k,v in train_res.items()},
             **{'valid_'+k:v for k,v in valid_res.items()},
             'lr_classifier':opt.param_groups[-1]['lr'],
             'lr_backbone':opt.param_groups[0]['lr'] if finetune else 0.0}
        hist.append(row)
        pd.DataFrame(hist).to_csv(exp_dir/'history_all_epochs.csv',index=False,encoding='utf-8-sig')
        save_checkpoint(last_path,model,epoch,valid_res,name,strategy,finetune,augmentation)
        if valid_res['f1_macro']>best_f1+1e-8:
            best_f1=valid_res['f1_macro']
            best_epoch=epoch
            without_improve=0
            save_checkpoint(best_path,model,epoch,valid_res,name,strategy,finetune,augmentation)
        else:
            without_improve+=1
        print(f'{name} E{epoch:02d} train_loss={train_res["loss"]:.4f} '
              f'valid_loss={valid_res["loss"]:.4f} '
              f'val_acc={valid_res["accuracy"]:.4f} val_F1macro={valid_res["f1_macro"]:.4f}')
        if without_improve>=EARLY_STOP_PATIENCE:
            print('Early stopping after',epoch,'epochs')
            break
    runtime=time.perf_counter()-start
    plot_history(pd.DataFrame(hist),exp_dir/'learning_curves.png',strategy)
    metadata={'experiment':name,'strategy':strategy,'augmentation':augmentation,
              'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),
              'trained_epochs':len(hist),'best_val_epoch':best_epoch,
              'best_val_f1_macro':best_f1,'train_seconds':runtime,
              'best_checkpoint':str(best_path),'last_checkpoint':str(last_path)}
    (exp_dir/'training_info.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
    all_runs.append(metadata)
    del model,train_loader,valid_loader,opt,scheduler,scaler
    if DEVICE.type=='cuda': torch.cuda.empty_cache()

pd.DataFrame(all_runs).to_csv(OUT/'training_summary_3_experiments.csv',index=False,encoding='utf-8-sig')
print('Đã huấn luyện và lưu checkpoints cho cả 3 thí nghiệm.')

## 6. Đánh giá đầy đủ từng thí nghiệm trên Validation **và** Test

Cho **cả ba mô hình**, lưu Precision/Recall/F1 theo từng lớp (4 chữ số thập phân), accuracy, macro/weighted avg, support, ma trận nhầm lẫn thô + chuẩn hóa, heatmap, mỗi ảnh và **4 xác suất Softmax**. Không sử dụng Test để chọn checkpoint hoặc chọn mô hình thắng cuộc.

In [ ]:
def load_own_checkpoint(exp_dir,finetune):
    model=build_model(finetune).to(DEVICE)
    checkpoint=torch.load(exp_dir/'checkpoints'/'best_validation_f1.pth',map_location='cpu',weights_only=False)
    assert checkpoint['manifest_sha256']==EXPECTED_MANIFEST_SHA256
    assert checkpoint['preprocessing']=='resize_preserve_aspect_pad'
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model,checkpoint

def save_confusion_matrices(y_true,y_pred,destination,prefix):
    cm=confusion_matrix(y_true,y_pred,labels=list(range(4)))
    cm_norm=cm/cm.sum(axis=1,keepdims=True).clip(min=1)
    pd.DataFrame(cm,index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(
        destination/f'{prefix}_confusion_matrix_counts.csv',encoding='utf-8-sig')
    pd.DataFrame(cm_norm,index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(
        destination/f'{prefix}_confusion_matrix_normalized.csv',encoding='utf-8-sig')
    for normalized,mat in [(False,cm),(True,cm_norm)]:
        fig,ax=plt.subplots(figsize=(7,6))
        im=ax.imshow(mat,cmap='Blues',vmin=0,vmax=1 if normalized else None)
        ax.set_xticks(range(4),CLASS_NAMES,rotation=30,ha='right')
        ax.set_yticks(range(4),CLASS_NAMES)
        ax.set(xlabel='Predicted label',ylabel='True label',title=f'{prefix} Confusion Matrix'+(' (Normalized)' if normalized else ''))
        maxval=max(np.max(mat),1e-9)
        for i in range(4):
            for j in range(4):
                ax.text(j,i,f'{mat[i,j]:.1%}' if normalized else str(int(mat[i,j])),
                        ha='center',va='center',color='white' if mat[i,j]>maxval/2 else '#102d44')
        fig.colorbar(im,ax=ax,fraction=.046)
        fig.tight_layout()
        fig.savefig(destination/(f'{prefix}_confusion_matrix'+('_normalized' if normalized else '')+'.png'),dpi=180,bbox_inches='tight')
        plt.close(fig)
    return cm

def export_evaluation(exp_dir,name,strategy,split,stats):
    true=np.asarray(stats.pop('_y_true'),dtype=int)
    pred=np.asarray(stats.pop('_y_pred'),dtype=int)
    probs=np.asarray(stats.pop('_probabilities'),dtype=float)
    paths=stats.pop('_paths')
    prefix=split
    report_text=classification_report(true,pred,labels=list(range(4)),target_names=CLASS_NAMES,digits=4,zero_division=0)
    (exp_dir/f'{prefix}_classification_report.txt').write_text(report_text,encoding='utf-8')
    report_dict=classification_report(true,pred,labels=list(range(4)),target_names=CLASS_NAMES,output_dict=True,zero_division=0)
    (exp_dir/f'{prefix}_classification_report.json').write_text(json.dumps(report_dict,ensure_ascii=False,indent=2),encoding='utf-8')
    report_df=pd.DataFrame(report_dict).T
    report_df.to_csv(exp_dir/f'{prefix}_classification_report.csv',encoding='utf-8-sig',float_format='%.6f')
    cm=save_confusion_matrices(true,pred,exp_dir,prefix)
    pred_df=pd.DataFrame({'path':paths, 'true_label_id':true,
        'true_label':[CLASS_NAMES[x] for x in true],
        'predicted_label_id':pred,
        'predicted_label':[CLASS_NAMES[x] for x in pred],
        'correct':(true==pred).astype(int)})
    for i,cl in enumerate(CLASS_NAMES): pred_df[f'prob_{cl}']=probs[:,i]
    pred_df.to_csv(exp_dir/f'{prefix}_predictions_probabilities.csv',index=False,encoding='utf-8-sig',float_format='%.8f')
    summary={k:v for k,v in stats.items() if isinstance(v,(float,int,np.floating,np.integer))}
    summary.update({'experiment':name,'strategy':strategy,'split':split,'support':len(true)})
    (exp_dir/f'{prefix}_summary_metrics.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
    print(f'[{name} | {split}] support={len(true)} accuracy={summary["accuracy"]:.4f} '
          f'macro P={summary["precision_macro"]:.4f} R={summary["recall_macro"]:.4f} F1={summary["f1_macro"]:.4f}')
    print(report_text)
    return summary,report_df,pred_df

report_rows=[]
per_class_rows=[]
run_predictions={}
for name,strategy,ft,aug in EXPERIMENTS:
    exp_dir=OUT/name
    model,checkpoint=load_own_checkpoint(exp_dir,ft)
    run_predictions[name]={}
    for split in ('valid','test'):
        loader=make_loader(split,eval_transform,shuffle=False)
        stats=pass_epoch(model,loader,collect_predictions=True)
        summary,report_df,pred_df=export_evaluation(exp_dir,name,strategy,split,stats)
        report_rows.append(summary)
        run_predictions[name][split]=pred_df
        for cl in CLASS_NAMES:
            d=report_df.loc[cl]
            per_class_rows.append({'experiment':name,'strategy':strategy,'split':split,'class_name':cl,
                'precision':float(d['precision']),'recall':float(d['recall']),
                'f1_score':float(d['f1-score']),'support':int(d['support'])})
        del loader
    del model
    if DEVICE.type=='cuda': torch.cuda.empty_cache()

metrics_summary_df=pd.DataFrame(report_rows)
metrics_summary_df.to_csv(OUT/'all_3_models_valid_test_summary.csv',index=False,encoding='utf-8-sig',float_format='%.8f')
per_class_df=pd.DataFrame(per_class_rows)
per_class_df.to_csv(OUT/'all_3_models_per_class_metrics.csv',index=False,encoding='utf-8-sig',float_format='%.8f')
print('==== Tổng hợp Test (không xếp hạng) ====')
print(metrics_summary_df.query('split=="test"')[['experiment','accuracy','precision_macro','recall_macro','f1_macro','precision_weighted','recall_weighted','f1_weighted','support']].to_string(index=False))

## 7. Biểu đồ so sánh — trình bày **cả ba**, không chọn model thắng

Biểu đồ các chỉ số từ **cùng Test 378 ảnh** và F1 từng lớp. Các file CSV giữ nguyên độ chính xác số học để copy vào báo cáo.

In [ ]:
from matplotlib.ticker import PercentFormatter

test_metrics=metrics_summary_df[metrics_summary_df.split=='test'].copy()
metrics_cols=['accuracy','precision_macro','recall_macro','f1_macro']
metric_names=['Accuracy','Precision Macro','Recall Macro','F1 Macro']
fig,ax=plt.subplots(figsize=(12,5))
x=np.arange(3)
width=0.19
for j,(col,label) in enumerate(zip(metrics_cols,metric_names)):
    ys=test_metrics.set_index('experiment').loc[[n for n,_,_,_ in EXPERIMENTS],col].values
    ax.bar(x+(j-1.5)*width,ys,width,label=label)
ax.set_xticks(x,[n.replace('mobilenet_','MobileNet ') for n,_,_,_ in EXPERIMENTS])
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylabel('Score on Test')
ax.set_ylim(0,1.04)
ax.legend(ncol=2)
ax.grid(axis='y',alpha=.18)
fig.tight_layout()
fig.savefig(OUT/'comparison_3_models_test_metrics.png',dpi=180,bbox_inches='tight')
plt.show()

fig,ax=plt.subplots(figsize=(12,5))
width=.24
cls_ix=np.arange(4)
for i,(name,_,_,_) in enumerate(EXPERIMENTS):
    f1=per_class_df.query('split=="test" and experiment==@name').set_index('class_name').loc[CLASS_NAMES,'f1_score'].values
    ax.bar(cls_ix+(i-1)*width, f1, width, label=name)
ax.set_xticks(cls_ix,CLASS_NAMES)
ax.set_ylim(0,1.04)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylabel('Test F1 per class')
ax.legend()
ax.grid(axis='y',alpha=.18)
fig.tight_layout()
fig.savefig(OUT/'comparison_3_models_f1_per_class.png',dpi=180,bbox_inches='tight')
plt.show()

## 8. Feature maps và Grad-CAM — xuất riêng **từng** MobileNet

- Feature maps tại bốn giai đoạn CNN (tầng đầu, giữa, sâu, trước GAP).
- Grad-CAM nhấn mạnh vùng ảnh tác động tới **logit của lớp dự đoán**, lưu 1 ảnh đại diện mỗi lớp thật cho mỗi mô hình. Nếu không có ảnh dự đoán đúng cho một lớp, dùng ảnh đầu tiên trong lớp và ghi rõ dự đoán.
- Kết quả trực quan hóa chỉ hỗ trợ diễn giải, không chứng minh mô hình nhìn đúng vùng trong mọi tình huống.

In [ ]:
def sample_path_for_class(name,class_name):
    preds=run_predictions[name]['test']
    sub=preds[(preds.true_label==class_name)&(preds.correct==1)]
    if len(sub)==0:sub=preds[preds.true_label==class_name]
    if len(sub)==0:return None
    row=sub.iloc[0]
    return row

def input_from_relpath(relpath):
    row=manifest[manifest.path==relpath]
    if len(row)!=1: raise RuntimeError('Không xác định được ảnh: '+str(relpath))
    with Image.open(row.iloc[0].resolved_path) as im:
        x=eval_transform(im.convert('RGB'))
    return x.unsqueeze(0).to(DEVICE)

def export_feature_maps(model,path,output_path):
    layer_ids=[0,4,10,18]
    acts={}
    hooks=[]
    for layer_id in layer_ids:
        def hook(m,ins,outs,idx=layer_id): acts[idx]=outs.detach().cpu()
        hooks.append(model.features[layer_id].register_forward_hook(hook))
    x=input_from_relpath(path)
    with torch.no_grad(): _=model(x)
    for h in hooks:h.remove()
    fig,axes=plt.subplots(4,7,figsize=(12,7.8))
    for ri,idx in enumerate(layer_ids):
        f=acts[idx][0].numpy()
        for ci in range(7):
            image=f[ci] if ci<len(f) else np.zeros_like(f[0])
            axes[ri,ci].imshow(image,cmap='viridis')
            axes[ri,ci].axis('off')
            if ci==0: axes[ri,ci].set_title(f'features[{idx}]\n{f.shape[1]}×{f.shape[2]}×{f.shape[0]}',fontsize=9,loc='left')
    fig.suptitle('Feature Maps | '+Path(path).name,fontsize=11)
    fig.tight_layout()
    fig.savefig(output_path,dpi=150,bbox_inches='tight')
    plt.close(fig)

def single_gradcam(model,path,target_class_id):
    x=input_from_relpath(path).requires_grad_(True)  # Cần grad qua backbone đã freeze khi tính Grad-CAM
    captured={}
    def on_forward(module,inputs,output):
        captured['activation']=output
        output.retain_grad()
    hook=model.features[18].register_forward_hook(on_forward)
    model.zero_grad(set_to_none=True)
    with torch.enable_grad():
        logits=model(x)
        probs=torch.softmax(logits,dim=1).detach().cpu().numpy()[0]
        logit=logits[0,int(target_class_id)]
        logit.backward()
    hook.remove()
    act=captured['activation'].detach()[0]
    grads=captured['activation'].grad.detach()[0]
    cam=torch.relu((grads.mean(dim=(1,2),keepdim=True)*act).sum(dim=0))
    cam=cam-cam.min()
    cam=cam/(cam.max()+1e-8)
    cam=torch.nn.functional.interpolate(cam[None,None],size=(IMAGE_SIZE,IMAGE_SIZE),mode='bilinear',align_corners=False)[0,0]
    return unnormalize(x[0]),cam.detach().cpu().numpy(),probs

for name,strategy,ft,aug in EXPERIMENTS:
    model,_=load_own_checkpoint(OUT/name,ft)
    if RUN_FEATURE_MAPS:
        selected=sample_path_for_class(name,'Green')
        if selected is not None:
            export_feature_maps(model,selected['path'],OUT/name/'feature_maps.png')
    if RUN_GRADCAM:
        fig,axes=plt.subplots(2,4,figsize=(14,7))
        for j,cl in enumerate(CLASS_NAMES):
            row=sample_path_for_class(name,cl)
            if row is None:continue
            pred_id=int(row['predicted_label_id'])
            image,cam,probs=single_gradcam(model,row['path'],pred_id)
            cmap=plt.get_cmap('jet')(cam)[...,:3]
            overlay=np.clip(.60*image+.40*cmap,0,1)
            axes[0,j].imshow(image)
            axes[1,j].imshow(overlay)
            axes[0,j].set_title(f'True: {cl}\nPred: {CLASS_NAMES[pred_id]}',fontsize=10)
            axes[1,j].set_title(f'Grad-CAM for pred\np={probs[pred_id]:.3f}',fontsize=10)
            for a in (axes[0,j],axes[1,j]):a.axis('off')
        fig.suptitle('Grad-CAM | '+name)
        fig.tight_layout()
        fig.savefig(OUT/name/'gradcam_each_class.png',dpi=170,bbox_inches='tight')
        plt.close(fig)
    del model
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
print('Đã lưu Feature Maps / Grad-CAM riêng theo từng thí nghiệm (nếu bật).')

## 9. RGB → Grayscale (ablation tại thời điểm suy luận, **không train lại**)

Tất cả model đã huấn luyện bằng RGB. Lần kiểm tra này chuyển ảnh Test sang grayscale **lặp lại thành 3 kênh**, giữ nguyên weights. Nó đo độ nhạy với việc loại bỏ thông tin màu, **không phải** so sánh công bằng với mô hình được huấn luyện riêng trên Grayscale. Không sử dụng kết quả này để chọn một mô hình tốt nhất.

In [ ]:
if RUN_GRAYSCALE_ABLATION:
    ablation_rows=[]
    for name,strategy,ft,aug in EXPERIMENTS:
        model,_=load_own_checkpoint(OUT/name,ft)
        gray_loader=make_loader('test',gray_eval_transform,shuffle=False)
        gray_stats=pass_epoch(model,gray_loader,collect_predictions=True)
        summary,report_df,pred_df=export_evaluation(OUT/name,name,strategy,'test_grayscale_inference_only',gray_stats)
        ablation_rows.append(summary)
        del model,gray_loader
        if DEVICE.type=='cuda':torch.cuda.empty_cache()
    pd.DataFrame(ablation_rows).to_csv(OUT/'rgb_vs_grayscale_inference_ablation.csv',index=False,encoding='utf-8-sig',float_format='%.8f')
else:
    print('Grayscale ablation đã tắt trong cấu hình.')

## 10. Tổng kết file output — **không nén ZIP, không chọn mô hình thắng cuộc**

Mỗi thí nghiệm lưu 2 checkpoint (best validation epoch + epoch cuối) và tất cả bảng đánh giá độc lập. So sánh giữa ba thí nghiệm được xuất **đầy đủ**, không có thao tác chọn hay xóa mô hình nào. Khi Save & Run All trên Kaggle, lấy dữ liệu trực tiếp từ phần Output.

In [ ]:
run_config={
    'seed':SEED, 'input_size':IMAGE_SIZE, 'padding_color':PAD_COLOR,
    'geometry_transform':'bilinear_preserve_aspect_then_center_pad_no_crop',
    'imagenet_normalization':{'mean':MEAN,'std':STD},
    'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'patience':EARLY_STOP_PATIENCE,
    'head_lr_frozen':HEAD_LR_FROZEN,'head_lr_finetune':HEAD_LR_FINETUNE,
    'backbone_lr_finetune':BACKBONE_LR_FINETUNE,'weight_decay':WEIGHT_DECAY,
    'pretrained_weights':'MobileNet_V2_Weights.IMAGENET1K_V1',
    'finetune_feature_indices':[15,16,17,18],
    'experiment_3_augmentation':'horizontal flip p=0.5 + affine deg=10 translate=.03 scale=[.96,1.0] on Train only',
    'model_selection_across_experiments':False,
    'epoch_checkpoint_selection':'own best validation F1-Macro; last epoch also retained',
    'evaluated_splits':['valid','test'],
    'manifest_sha256':EXPECTED_MANIFEST_SHA256,
    'experiment_names':[item[0] for item in EXPERIMENTS],
    'device':str(DEVICE),
    'torch_version':torch.__version__,
}
(OUT/'run_config.json').write_text(json.dumps(run_config,ensure_ascii=False,indent=2),encoding='utf-8')
all_files=sorted(p.relative_to(OUT).as_posix() for p in OUT.rglob('*') if p.is_file())
pd.DataFrame({'relative_path':all_files}).to_csv(OUT/'output_file_index.csv',index=False,encoding='utf-8-sig')
print('LƯU TOÀN BỘ THƯ MỤC (KHÔNG ZIP):',OUT)
print('Tổng số file:',len(all_files))
for p in all_files:
    print(' -',p)